# Assignment 2.4 — Manual Stable Diffusion CFG with `diffusers` (Kaggle P100)

**Goal:** manually connect Stable Diffusion components without using `StableDiffusionPipeline`.

We will:
1. Encode the real prompt → conditional CLIP embedding.
2. Encode `""` → unconditional CLIP embedding.
3. Start from the **same random latent** for every guidance scale.
4. Run DDIM manually.
5. At every timestep run the U-Net **twice**:
   - conditional forward pass
   - unconditional forward pass
6. Apply CFG:
\[
\epsilon = \epsilon_u + s(\epsilon_c-\epsilon_u)
\]
7. Decode the final latent with the VAE.
8. Compare `s = [1, 3, 5, 7.5, 12, 20]`.

> Recommended Kaggle settings: **GPU = Tesla P100**, Internet = ON.

## 0. P100-safe environment setup

Tesla P100 is Pascal (`sm_60`). Recent Kaggle images may ship a PyTorch build that no longer includes `sm_60`, which causes:

`CUDA error: no kernel image is available for execution on the device`

This cell runs **before importing torch**. It checks the current wheel in a subprocess. If `sm_60` is missing, it installs the known CUDA 12.6-era PyTorch 2.7.0 stack that includes `sm_60`.

It deliberately avoids `--force-reinstall`.

In [ ]:
import subprocess, sys, json


def probe_torch():
    script = r"""
import json
try:
    import torch
    print(json.dumps({
        "ok": True,
        "version": torch.__version__,
        "cuda": torch.version.cuda,
        "available": torch.cuda.is_available(),
        "archs": torch.cuda.get_arch_list() if torch.cuda.is_available() else [],
        "capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    }))
except Exception as e:
    print(json.dumps({"ok": False, "error": repr(e)}))
"""
    p = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
    line = p.stdout.strip().splitlines()[-1] if p.stdout.strip() else "{}"
    try:
        return json.loads(line)
    except Exception:
        return {"ok": False, "stdout": p.stdout, "stderr": p.stderr}


info = probe_torch()
print("Before setup:", info)

# Only Pascal P100 (sm_60) needs the compatibility wheel. T4 and newer GPUs
# keep Kaggle's preinstalled PyTorch so the current kernel is not disrupted.
is_p100 = tuple(info.get("capability") or []) == (6, 0)
needs_p100_torch = is_p100 and (not info.get("ok", False) or "sm_60" not in info.get("archs", []))

if needs_p100_torch:
    print("\nInstalling P100-compatible PyTorch 2.7.0 + CUDA 12.6 ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "torch==2.7.0", "torchvision==0.22.0", "torchaudio==2.7.0",
        "--index-url", "https://download.pytorch.org/whl/cu126"
    ])
else:
    print("\nKeeping Kaggle's existing PyTorch wheel for this GPU.")

print("\nInstalling assignment dependencies ...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.35,<0.36",
    "transformers>=4.53,<5",
    "accelerate>=1.8",
    "safetensors>=0.5",
    "huggingface_hub>=0.33",
    "Pillow>=10,<12"
])

## 1. Verify GPU and imports

Expected on P100:
- GPU capability: `(6, 0)`
- `sm_60` must appear in `torch.cuda.get_arch_list()`.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Capability:', torch.cuda.get_device_capability(0))
print('Supported arch:', torch.cuda.get_arch_list())
print('A Kaggle CUDA GPU is required; both P100 and T4 are supported for this notebook.')

## 2. Load Stable Diffusion components separately

No high-level pipeline is used.

We load:
- tokenizer
- CLIP text encoder
- U-Net
- VAE
- DDIM scheduler

In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DEVICE = "cuda"
DTYPE = torch.float16

tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder", torch_dtype=DTYPE
).to(DEVICE)
vae = AutoencoderKL.from_pretrained(
    MODEL_ID, subfolder="vae", torch_dtype=DTYPE
).to(DEVICE)
unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID, subfolder="unet", torch_dtype=DTYPE
).to(DEVICE)
scheduler = DDIMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

text_encoder.eval()
vae.eval()
unet.eval()

for m in (text_encoder, vae, unet):
    for p in m.parameters():
        p.requires_grad_(False)

print("Loaded all components manually.")

## 3. Prompt → conditional / unconditional embeddings

Exactly as required:
- conditional prompt = real prompt
- unconditional prompt = `""`

In [ ]:
PROMPT = "a cinematic photograph of a red vintage sports car parked on a rainy Tokyo street at night"

def encode_text(text: str):
    tokens = tokenizer(
        text,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    input_ids = tokens.input_ids.to(DEVICE)
    with torch.no_grad():
        emb = text_encoder(input_ids)[0]
    return emb

cond_emb = encode_text(PROMPT)
uncond_emb = encode_text("")

print("cond_emb:", tuple(cond_emb.shape))
print("uncond_emb:", tuple(uncond_emb.shape))

## 4. Create one initial latent and reuse it for every scale

This is essential for a fair comparison.  
Only the guidance scale changes.

In [ ]:
HEIGHT = 512
WIDTH = 512
SEED = 2026
NUM_STEPS = 30

latent_h = HEIGHT // 8
latent_w = WIDTH // 8
latent_channels = unet.config.in_channels

gen = torch.Generator(device=DEVICE).manual_seed(SEED)
base_latent = torch.randn(
    (1, latent_channels, latent_h, latent_w),
    generator=gen,
    device=DEVICE,
    dtype=DTYPE,
)

print("Initial latent:", tuple(base_latent.shape))

## 5. Manual DDIM inference loop

This is the core of Assignment 2.4.

At each timestep:
1. `U-Net(z_t, t, cond)` → conditional noise
2. `U-Net(z_t, t, uncond)` → unconditional noise
3. CFG
4. DDIM scheduler step

In [ ]:
@torch.inference_mode()
def generate_manual_cfg(guidance_scale: float):
    scheduler.set_timesteps(NUM_STEPS, device=DEVICE)

    # exact same starting noise for every experiment
    latents = base_latent.clone()

    # DDIM uses init_noise_sigma from scheduler abstraction
    latents = latents * scheduler.init_noise_sigma

    for t in tqdm(
        scheduler.timesteps,
        desc=f"CFG={guidance_scale}",
        leave=False
    ):
        latent_model_input = scheduler.scale_model_input(latents, t)

        # REQUIRED: two separate U-Net forward passes
        uncond_noise = unet(
            latent_model_input,
            t,
            encoder_hidden_states=uncond_emb
        ).sample

        cond_noise = unet(
            latent_model_input,
            t,
            encoder_hidden_states=cond_emb
        ).sample

        # Classifier-Free Guidance
        noise = uncond_noise + guidance_scale * (cond_noise - uncond_noise)

        # DDIM: z_t -> z_{t-1}
        latents = scheduler.step(
            noise,
            t,
            latents,
            eta=0.0
        ).prev_sample

    # Stable Diffusion latent scaling before VAE decode
    scaling = vae.config.scaling_factor
    decoded = vae.decode(latents / scaling).sample

    image = (decoded / 2 + 0.5).clamp(0, 1)
    image = image.detach().float().cpu()
    image = image.permute(0, 2, 3, 1).numpy()[0]
    image = (image * 255).round().astype(np.uint8)
    return Image.fromarray(image)

## 6. Run all requested guidance scales

In [ ]:
GUIDANCE_SCALES = [1, 3, 5, 7.5, 12, 20]

results = {}
for s in GUIDANCE_SCALES:
    results[s] = generate_manual_cfg(float(s))
    torch.cuda.empty_cache()

print("Done.")

## 7. Visualize results side-by-side

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, s in zip(axes.flat, GUIDANCE_SCALES):
    ax.imshow(results[s])
    ax.set_title(f"Guidance scale = {s}")
    ax.axis("off")

plt.suptitle(PROMPT, fontsize=12)
plt.tight_layout()
plt.show()

## 8. Save images

Use the grid in your report and discuss:
- low guidance: weak prompt adherence
- middle guidance: stronger prompt adherence + natural image quality
- excessive guidance: oversaturation / harsh contrast / artifacts / less natural texture

Do **not** assume 7.5 is automatically the sweet spot. Identify it from your actual outputs.

In [ ]:
from pathlib import Path

save_dir = Path("/kaggle/working/assignment_2_4_outputs")
save_dir.mkdir(parents=True, exist_ok=True)

for s, img in results.items():
    name = str(s).replace(".", "_")
    img.save(save_dir / f"cfg_{name}.png")

print("Saved to:", save_dir)

## 9. Conclusion from the generated images

| Guidance scale | Observation |
|---:|---|
| 1 | Very weak prompt adherence: mostly a monochrome rainy street; the red vintage car is not clear. |
| 3 | The scene begins to follow the prompt, but colour and subject detail are still weak. |
| 5 | Car and street composition become more recognizable, with moderate prompt adherence. |
| 7.5 | A coherent image with stronger prompt adherence, but less vivid detail than the best result. |
| 12 | Best balance: a clear red vintage sports car, neon Tokyo-street atmosphere, wet reflections and natural composition. |
| 20 | Prompt elements are strong, but the red/contrast become too harsh and the image looks less natural. |

**Sweet spot:** `s = 12`

**First clear degradation:** `s = 20`

**Reason:** With a very high guidance scale, the conditional update dominates the model prior, producing over-saturation and harsh contrast.